# Audit kualitas anotasi gegenpressing

Notebook ini memeriksa integritas label dan membuat review queue untuk label
`Ragu` atau confidence rendah. Review queue sengaja tidak memuat outcome quick
regain agar peninjauan ulang tidak mengubah label perilaku berdasarkan hasil.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.annotation_audit import build_annotation_audit
from src.research_readiness import build_model_manifest

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'annotation_audit_current'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
annotations = pd.read_csv(
    PROCESSED_DIR / 'gegenpressing_annotation_manifest.csv'
)
episodes = pd.read_csv(PROCESSED_DIR / 'episode_manifest.csv')
model_manifest = build_model_manifest(episodes, annotations)

summary, review_queue, checks = build_annotation_audit(
    annotations,
    model_manifest,
)
summary.to_csv(OUTPUT_DIR / 'annotation_audit_summary.csv', index=False)
review_queue.to_csv(OUTPUT_DIR / 'annotation_review_queue.csv', index=False)
(OUTPUT_DIR / 'annotation_audit_checks.json').write_text(
    json.dumps({key: bool(value) for key, value in checks.items()}, indent=2),
    encoding='utf-8',
)

print('Quality checks:')
display(checks.to_frame())
print('Summary by match and split:')
display(summary)
print('Blinded review queue:', len(review_queue))
display(review_queue)
print('Artifacts:', OUTPUT_DIR)
if checks['all_annotations_completed']:
    print('Semua sampel sudah dianotasi; tinjau review queue sebelum training final.')
else:
    print('Audit sementara. Kembali ke notebook 08 untuk menyelesaikan queue.')

Quality checks:


,passed
all_annotations_completed,True
all_labels_valid,True
candidate_id_unique,True
confidence_complete,True
confidence_valid,True
uncertain_have_notes,True
annotator_complete,True


Summary by match and split:


,split,match_id,queue,completed,usable,positive,negative,uncertain,low_confidence
0,test,J03WPY,323,323,323,256,67,0,0
1,train,J03WMX,408,408,408,341,67,0,5
2,train,J03WN1,33,33,33,27,6,0,0
3,train,J03WOH,298,298,298,262,36,0,1
4,train,J03WOY,342,342,342,296,46,0,0
5,train,J03WR9,274,274,274,236,38,0,0
6,validation,J03WQQ,195,195,195,163,32,0,0


Blinded review queue: 6


,candidate_id,match_id,split,period_id,loss_timestamp,loss_frame,is_gegenpressing,annotation_confidence,annotation_notes,annotated_at_utc,annotator,review_reason
0,J03WMX_P1_F12133,J03WMX,train,1,0 days 00:01:25.320000,12133,1,1,NaN,2026-10-26T14:13:08.934406+00:00,rizal,low_confidence
1,J03WMX_P1_F72996,J03WMX,train,1,0 days 00:41:59.840000,72996,1,1,NaN,2026-10-26T14:50:54.564451+00:00,rizal,low_confidence
2,J03WMX_P1_F73826,J03WMX,train,1,0 days 00:42:33.040000,73826,0,1,NaN,2026-10-26T15:37:24.877780+00:00,rizal,low_confidence
3,J03WMX_P1_F73879,J03WMX,train,1,0 days 00:42:35.160000,73879,0,1,NaN,2026-10-26T15:37:36.802847+00:00,rizal,low_confidence
4,J03WMX_P2_F119897,J03WMX,train,2,0 days 00:13:15.880000,119897,1,1,NaN,2026-10-26T15:40:59.985341+00:00,rizal,low_confidence
5,J03WOH_P2_F103255,J03WOH,train,2,0 days 00:02:10.200000,103255,0,1,NaN,2026-10-27T08:05:07.478700+00:00,rizal,low_confidence


Artifacts: c:\VAEP-Track\artifacts\annotation_audit_current
Semua sampel sudah dianotasi; tinjau review queue sebelum training final.
